<a href="https://colab.research.google.com/github/everestso/AI-Education/blob/main/c264s26_Gymnasium_dqn_pong.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
import os
from datetime import datetime

drive.mount("/content/drive")

save_dir = "/content/drive/MyDrive/RL_Models"
os.makedirs(save_dir, exist_ok=True)

Mounted at /content/drive


In [ ]:
# Install packages
!pip uninstall -y gym
!pip install -U "gymnasium[atari]" "stable-baselines3[extra]"

# Register Atari environments
import gymnasium as gym
import ale_py

gym.register_envs(ale_py)

# Verify registration
atari_envs = [env_id for env_id in gym.envs.registry.keys() if env_id.startswith("ALE/")]
print("Number of ALE envs:", len(atari_envs))
print("Sample envs:", atari_envs[:10])

# Smoke test
env = gym.make("ALE/Breakout-v5")
obs, info = env.reset()
print("Observation shape:", obs.shape)
env.close()

Found existing installation: gym 0.25.2
Uninstalling gym-0.25.2:
  Successfully uninstalled gym-0.25.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 117.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.5/187.5 kB 24.0 MB/s eta 0:00:00
Number of ALE envs: 104
Sample envs: ['ALE/Adventure-v5', 'ALE/AirRaid-v5', 'ALE/Alien-v5', 'ALE/Amidar-v5', 'ALE/Assault-v5', 'ALE/Asterix-v5', 'ALE/Asteroids-v5', 'ALE/Atlantis2-v5', 'ALE/Atlantis-v5', 'ALE/Backgammon-v5']
Observation shape: (210, 160, 3)


## Atari / Gymnasium Documentation

- [ALE / Atari Getting Started](https://ale.farama.org/getting-started/)
- [ALE Environment Reference](https://ale.farama.org/environments/)
- [Gymnasium Registry Documentation](https://gymnasium.farama.org/api/registry/)

# Test 1

In [ ]:
import gymnasium as gym
import ale_py

gym.register_envs(ale_py)

from stable_baselines3.common.env_util import make_atari_env
from stable_baselines3.common.vec_env import VecFrameStack
from stable_baselines3 import A2C

# 4 parallel Atari envs with standard preprocessing
vec_env = make_atari_env(
    "ALE/Pong-v5",   # if make_atari_env in your setup prefers old ids, use PongNoFrameskip-v4
    n_envs=4,
    seed=0,
    wrapper_kwargs=dict(terminal_on_life_loss=False),
)
vec_env = VecFrameStack(vec_env, n_stack=4)


In [ ]:
#model = A2C("CnnPolicy", vec_env, verbose=1)


Streaming output truncated to the last 5000 lines.
|    iterations         | 500      |
|    time_elapsed       | 36       |
|    total_timesteps    | 170000   |
| train/                |          |
|    entropy_loss       | -0.487   |
|    explained_variance | 0.911    |
|    learning_rate      | 0.0007   |
|    n_updates          | 8499     |
|    policy_loss        | 0.00271  |
|    value_loss         | 0.0228   |
------------------------------------
Saved checkpoint: a2c_pong_170000.zip
------------------------------------
| rollout/              |          |
|    ep_len_mean        | 868      |
|    ep_rew_mean        | -20.6    |
| time/                 |          |
|    fps                | 272      |
|    iterations         | 100      |
|    time_elapsed       | 7        |
|    total_timesteps    | 172000   |
| train/                |          |
|    entropy_loss       | -0.467   |
|    explained_variance | 0.838    |
|    learning_rate      | 0.0007   |
|    n_updates         

KeyboardInterrupt: 

In [ ]:
from stable_baselines3 import DQN

model = DQN(
    "CnnPolicy",
    vec_env,
    verbose=1,
    buffer_size=100000,
    learning_starts=10000,
    batch_size=32,
    train_freq=4,
    target_update_interval=1000,
)

Using cuda device
Wrapping the env in a VecTransposeImage.


In [ ]:
chunk_size = 20_000
num_chunks = 100

for i in range(1, num_chunks + 1):
    model.learn(
        total_timesteps=chunk_size,
        reset_num_timesteps=(i == 1),
        log_interval=100
    )

    total_so_far = i * chunk_size

    model_path = os.path.join(save_dir, f"dqn_pong_model_{total_so_far}")
    model.save(model_path)
    print("Model saved to:", model_path + ".zip")

Model saved to: /content/drive/MyDrive/RL_Models/dqn_pong_model_20000.zip
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 826      |
|    ep_rew_mean      | -20.8    |
|    exploration_rate | 0.05     |
| time/               |          |
|    episodes         | 100      |
|    fps              | 291      |
|    time_elapsed     | 1        |
|    total_timesteps  | 20384    |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.056    |
|    n_updates        | 648      |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 777      |
|    ep_rew_mean      | -21      |
|    exploration_rate | 0.05     |
| time/               |          |
|    episodes         | 200      |
|    fps              | 286      |
|    time_elapsed     | 67       |
|    total_timesteps  | 39276    |
| train/              |          |
|    learning_ra

In [ ]:
### Continue Code
chunk_size = 20_000
num_chunks = 100
start_total = 5340000
for i in range(1, num_chunks + 1):
    model.learn(
        total_timesteps=chunk_size,
        reset_num_timesteps=False,
        log_interval=100
    )

    total_so_far = start_total + i * chunk_size

    model_path = os.path.join(save_dir, f"dqn_pong_model_{total_so_far}")
    model.save(model_path)
    print("Model saved to:", model_path + ".zip")

------------------------------------
| rollout/              |          |
|    ep_len_mean        | 2.97e+03 |
|    ep_rew_mean        | -18      |
| time/                 |          |
|    fps                | 277      |
|    iterations         | 100      |
|    time_elapsed       | 28       |
|    total_timesteps    | 5348000  |
| train/                |          |
|    entropy_loss       | -0.653   |
|    explained_variance | -0.497   |
|    learning_rate      | 0.0007   |
|    n_updates          | 67806    |
|    policy_loss        | -0.232   |
|    value_loss         | 0.506    |
------------------------------------
------------------------------------
| rollout/              |          |
|    ep_len_mean        | 2.96e+03 |
|    ep_rew_mean        | -18      |
| time/                 |          |
|    fps                | 289      |
|    iterations         | 200      |
|    time_elapsed       | 55       |
|    total_timesteps    | 5356000  |
| train/                |          |
|

In [ ]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
model_path = os.path.join(save_dir, f"dqn_pong_model_{timestamp}")
model.save(model_path)

print("Model saved to:", model_path + ".zip")

# Load and Continue

In [ ]:
# Path to saved model
model_path = "/content/drive/MyDrive/RL_Models/dqn_pong_model_5340000.zip"

if not os.path.exists(model_path):
    raise FileNotFoundError(f"Model file not found: {model_path}")

# Load model into variable name expected by later cells
model = DQN.load(model_path)
model.set_env(vec_env)

print("Loaded model:", model_path)

FileNotFoundError: Model file not found: /content/drive/MyDrive/RL_Models/dqn_pong_model_5340000.zip